코드1

In [1]:
from dotenv import load_dotenv
# 같은 위치의 .env 파일을 메모리에 로드하는 함수
load_dotenv(override=True)


True

코드2

In [ ]:
from openai import OpenAI
client = OpenAI()

print(client)

In [ ]:


completion = client.chat.completions.create(
  model="gpt-4.1-nano",
  messages=[
        {
            "role": "system",  ## 역할 부여 (시스템)
            "content": "너는 python 10년차 프로그래머다",  ## 페르소나를 정의 
        },
        {
            "role": "user",  ## 역할 부여 (사용자)
            "content": "피보나치 수열을 생성하는 파이썬 프로그램을 작성해줘", ## 사용자가 입력하는 질문
        }
  ]
)


## 답변을 출력
print(completion.choices[0].message.content)  

## 출력을 상세하게 보면 답변의  role='assistant'
# print(completion.choices[0])  


### 각종 파라미터를 추가하여 옵션을 줄수 있음

**model**
사용할(호출) LLM 모델 이름을 작성

**messages**
AI와 나눈 대화 내역입니다. 사용자와 AI의 메시지를 순서대로 담아 전달합니다.

**max_tokens**
AI가 한 번에 생성할 수 있는 최대 글자량(토큰)입니다. 너무 작으면 답변이 중간에 잘립니다.

**n**
같은 질문에 대해 몇 가지 답변을 생성할지 설정합니다. 기본값 1을 권장합니다. 숫자가 클수록 비용도 그만큼 늘어납니다.

**response_format**
응답 형식을 지정합니다. `{ "type": "json_object" }` 로 설정하면 JSON 형태로만 답변을 받을 수 있습니다.

**temperature**
답변의 창의성(무작위성)을 조절합니다. 0에 가까울수록 일관되고 정확한 답변, 1에 가까울수록 다양하고 창의적인 답변이 나옵니다. `top_p`와 동시에 조절하지 않는 것을 권장합니다.

**top_p**
temperature와 비슷하게 답변의 다양성을 조절하는 또 다른 방법입니다. 값이 낮을수록 확률이 높은 단어만 선택합니다. temperature와 둘 중 하나만 조절하는 것을 권장합니다.

### 스트리밍 출력

코드 3

In [ ]:
from openai import OpenAI

client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {
            "role": "system",
            "content": "당신은 파이썬 프로그래머입니다.",
        },
        {
            "role": "user",
            "content": "피보나치 수열을 생성하는 파이썬 프로그램을 작성해주세요.",
        },
    ],
    stream=True,  # 스트림 모드 활성화
)

final_answer = []

# 스트림 모드에서는 completion.choices 를 반복문으로 순회
for chunk in completion:
    print(chunk.choices[0].delta.content, end="")
        

### 연속적 대화

일반 대화

코드4

In [ ]:
completion = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant. You must answer in Korean.",
        },
        {
            "role": "user",
            "content": "대한민국의 수도는 어디인가요?",
        },
    ],
)

print(completion.choices[0].message.content)

함수화

코드5

In [ ]:
def ask(question):
    completion = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant. You must answer in Korean.",
            },
            {
                "role": "user",
                "content": question,  # 사용자의 질문을 입력
            },
        ],
    )
    # 답변을 반환
    return completion.choices[0].message.content

In [ ]:
# 첫 번째 질문
ask("대한민국의 수도는 어디인가요?")  

정상적인 연속대화가 안됨

In [ ]:
# 두 번째 질문
ask("영어로 답변해 주세요")

하드코딩을 하면 원하는 답변을 정상적으로 받을 수 있음

코드6

In [ ]:
completion = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant. You must answer in Korean.",
        },
        {
            "role": "user",
            "content": "대한민국의 수도는 어디인가요?",  # 첫 번째 질문
        },
        {
            "role": "assistant",
            "content": "대한민국의 수도는 서울입니다.",  # 첫 번째 답변
        },
        {
            "role": "user",
            "content": "이전의 답변을 영어로 번역해 주세요.",  # 두 번째 질문
        },
    ],
)

# 두 번째 답변을 출력
print(completion.choices[0].message.content)

위 내용을 함수형태로 구현 기존 messages=[] 부분을 따로 빼서 계속 쌓는형식으로 만든다

코드7

In [ ]:
def ask(question, message_history=[], model="gpt-4.1-nano"):
    if len(message_history) == 0:
        # 최초 질문
        message_history.append(
            {
                "role": "system",
                "content": "You are a helpful assistant. You must answer in Korean.",
            }
        )

    # 사용자 질문 추가
    message_history.append(
        {
            "role": "user",
            "content": question,
        },
    )

    # GPT에 질문을 전달하여 답변을 생성
    completion = client.chat.completions.create(
        model=model,
        messages=message_history,
    )

    # 사용자 질문에 대한 답변을 추가
    message_history.append(
        {"role": "assistant", "content": completion.choices[0].message.content}
    )

    return message_history

In [ ]:
# 최초 질문
message_history = ask("양자역학에 대해서 쉽게 설명해 주세요", message_history=[])
# 최초 답변
print(message_history[-1])

In [ ]:
message_history

In [ ]:
# 두 번째 질문
message_history = ask(
    "이전의 내용을 영어로 답변해 주세요", message_history=message_history
)
# 두 번째 답변
print(message_history[-1])

In [ ]:
message_history

# Json 형식 답변

json 형식으로 답변을 받으면 프로그램에서 후 처리(DB 입력, 화면출력 등)가 편하다

In [ ]:


completion = client.chat.completions.create(
  model="gpt-4.1-nano",
  messages=[
        {
            "role": "system",  ## 역할 부여 (시스템)
            "content": "json 형식으로 출력해줘",  ## 페르소나를 정의 
        },
        {
            "role": "user",  ## 역할 부여 (사용자)
            "content": "직장인 필수 영단어 주제로 4지선다형 객관식 문제를 만들어줘, 정답은 index 번호로 알려줘", ## 사용자가 입력하는 질문
        }
  ],
  response_format={ "type": "json_object" },
  max_tokens=1000,
)


## 답변을 출력
print(completion.choices[0].message.content)  




여러 문제

In [ ]:


completion = client.chat.completions.create(
  model="gpt-4.1-nano",
  messages=[
        {
            "role": "system",  ## 역할 부여 (시스템)
            "content": "json 형식으로 출력해줘",  ## 페르소나를 정의 
        },
        {
            "role": "user",  ## 역할 부여 (사용자)
            "content": "직장인 필수 영단어 주제로 4지선다형 객관식 문제를 만들어줘, 정답은 index 번호로 알려줘", ## 사용자가 입력하는 질문
        }
  ],
  response_format={ "type": "json_object" },
  max_tokens=5000,
  n=3
)


#len(completion.choices)
# # choices[0]가 n만큼 나온다
for res in completion.choices:
    print(res.message.content)

# 명확하게 구조화

In [ ]:


completion = client.chat.completions.create(
  model="gpt-4.1-nano",
  messages=[
        {
            "role": "system",  ## 역할 부여 (시스템)
            "content": "json 형식으로 출력해줘",  ## 페르소나를 정의 
        },
        {
            "role": "user",  ## 역할 부여 (사용자)
            "content": "직장인 필수 영단어 주제로 4지선다형 객관식 문제를 만들어줘, 정답은 index 번호로 알려줘"
            "json 구조는 아래에 따라 줘"
            """{ 
                "question": "문제",
                "choices": [
                    "보기1",
                    "보기2",
                    "보기3",
                    "보기4"
                ],
                "answer": N
                }
            """
        }
  ],
  response_format={ "type": "json_object" },
  max_tokens=5000,
  n=3
)


## 답변을 출력
# print(completion.choices[0].message.content)  


# # choices[0]가 n만큼 나온다
for res in completion.choices:
    print(res.message.content)



# JSON 형식의 답변을 파이썬 객체로 변환

In [ ]:
type(res.message.content)
#위 출력이 xml 처럼 보이지만 객체의 속성은 str 이다.

In [ ]:
import json

json_obj = json.loads(res.message.content) # 답변 1개
json_obj

In [ ]:
# LLM의 답변을 json 객체에 담아준다 
type(json_obj)

In [ ]:
# 모든 json 형식의 답변을 파이썬 객체로 변환
json_result = [json.loads(res.message.content) for res in completion.choices]
json_result

